# Train detector khung xoay (YOLO11s-OBB) trên Colab

Mục tiêu: đếm đúng chuối trên nải xòe (`naichuoi.jpeg`, ~20 quả; model C hiện ra 12). Runtime -> Change runtime type -> **T4 GPU**.

Kết quả lưu trên Google Drive (`MyDrive/fruit_runs/obb`), ngắt kết nối thì chạy lại từ ô 1: ô train tự resume từ `last.pt`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/fruit_runs/obb'
NAME = 'obb_colab'
!mkdir -p {RUNS}

Clone repo (repo ~3GB nên dùng `--depth 1`). Nếu repo private: thay URL bằng `https://<token>@github.com/toannd135/BTL_XLA.git` (token GitHub quyền đọc repo).

In [ ]:
%cd /content
!test -d BTL_XLA || git clone --depth 1 https://github.com/toannd135/BTL_XLA.git
%cd /content/BTL_XLA
!git pull --ff-only
!wc -l work/obb/data/train.txt work/obb/data/val.txt && ls work/obb/data/real/train/images | wc -l

In [ ]:
!pip -q install ultralytics==8.4.180

Train: 60 epoch, batch 16 (T4 16GB; OOM thì giảm 8). Tham số: `name epochs batch project`.

In [ ]:
!python work/obb/train_obb.py {NAME} 60 16 {RUNS}

Đo đếm quả: `naichuoi` (cần ~20 chuối), `neg` (71 ảnh nền, càng ít ảnh báo sai càng tốt), `synth_holdout`, `test`. So với model hiện tại (C, box thẳng) ở ô sau.

In [ ]:
BEST = f'{RUNS}/{NAME}/weights/best.pt'
!python work/obb/eval_count.py {BEST} {NAME} 0.4 0.6
!python work/obb/eval_count.py exports/det_C_candidate.pt C_hbb 0.4 0.6
!mkdir -p {RUNS}/eval && cp work/obb/eval/*.json {RUNS}/eval/

## Bảng và biểu đồ như README mục 5

In bảng 5.1 (mAP, AP50 từng loại, ảnh nền báo sai; A/B/C lấy số trong README), bảng 5.2 (P/R/F1 ở conf 0.40, độ trễ), bảng đếm quả, và vẽ hình (so sánh A/B/C/OBB, đánh đổi, đường huấn luyện, đếm quả, naichuoi). Lưu ở `RUNS/report` trên Drive; `tables.md` dán thẳng vào README được.

In [ ]:
REPORT = f'{RUNS}/report'
!python work/obb/report_obb.py {BEST} {RUNS}/{NAME} {REPORT}

In [ ]:
from pathlib import Path
from IPython.display import Markdown, Image, display
display(Markdown(Path(f'{REPORT}/tables.md').read_text()))
for f in sorted(Path(f'{REPORT}/figures').glob('*.png')):
    print(f.name); display(Image(str(f)))

Hình có sẵn của Ultralytics (README 5.6): confusion matrix và đường PR/F1 trên test, kết quả train theo epoch.

In [ ]:
vt = Path(f'{REPORT}/val_test')
for n in ['confusion_matrix_normalized.png', 'confusion_matrix.png', 'BoxPR_curve.png', 'BoxF1_curve.png']:
    if (vt / n).exists():
        print(n); display(Image(str(vt / n), width=700))
rp = Path(f'{RUNS}/{NAME}/results.png')
if rp.exists(): print(rp.name); display(Image(str(rp), width=900))

Vẽ kết quả trên `naichuoi.jpeg` để xem bằng mắt.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
r = YOLO(BEST).predict('naichuoi.jpeg', conf=0.4, iou=0.6, agnostic_nms=True, imgsz=640, max_det=300, verbose=False)[0]
print('so chuoi:', int((r.obb.cls == 0).sum()), '| tong box:', len(r.obb))
r.save(filename='/content/naichuoi_obb.jpg'); display(Image('/content/naichuoi_obb.jpg'))

Lưu model để đưa về máy: tải `det_obb_s.pt` từ Drive, đặt vào `exports/`, chạy backend với `MODEL_PATH=exports/det_obb_s.pt` (backend tự nhận model OBB).

In [ ]:
!cp {BEST} /content/drive/MyDrive/fruit_runs/det_obb_s.pt && ls -la /content/drive/MyDrive/fruit_runs/